# SignatureAI — Same-Dataset Siamese Verification (Kaggle)

This notebook fine-tunes the API-compatible **ResNet18 Siamese embedding model** using the native verification-pair form of the same signature corpus used to build the detection dataset.

- Pair source: `Mels22/SigDetectVerifyFlow`
- Inputs: `sample_signature` (reference) + `to_verify_signature` (candidate)
- Labels: `0 = genuine`, `1 = forged`
- Split: identity group when available; otherwise normalized reference-signature fingerprint
- Threshold: selected from validation only and applied once to test
- Export: plain ResNet18 state dict plus an S3 `latest` alias compatible with the current FastAPI service

## Kaggle settings

Enable **GPU** and **Internet** before Run All. Optional Kaggle secrets are `HF_TOKEN`, `COMET_API_KEY`, `COMET_WORKSPACE`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`, `AWS_REGION`, and `AWS_S3_BUCKET`.

> Important: if this dataset does not expose writer IDs, the fallback split prevents the exact same reference image from crossing splits, but it cannot prove complete writer-disjointness. The generated report records that limitation.

In [ ]:
# Cell 1 — install only packages that Kaggle may not already provide
%pip install -q "datasets>=2.19,<5" "huggingface_hub>=0.24" "comet_ml>=3.44" "boto3>=1.34" "scikit-learn>=1.4"

In [ ]:
# Cell 2 — imports, reproducibility and runtime check
import os
import json
import time
import random
import hashlib
import datetime
import uuid
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support

assert Path('/kaggle/working').exists(), 'Run this Kaggle notebook inside Kaggle.'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('WARNING: enable a Kaggle GPU before training.')

In [ ]:
# Cell 3 — experiment configuration
HF_PAIR_DATASET = 'Mels22/SigDetectVerifyFlow'
HF_REVISION = 'main'
LABEL_GENUINE = 0
LABEL_FORGED = 1
SPLIT_RATIOS = (0.80, 0.10, 0.10)
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
COSINE_MARGIN = 0.20
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4
PRETRAINED = True

USE_COMET = True
COMET_PROJECT = 'signature-ai-similarity'
UPLOAD_TO_S3 = True
S3_PREFIX = 'signatureai/similarity'
S3_ALIAS = 'latest'

RUN_VERSION = 'v' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT_DIR = Path('/kaggle/working/signature_similarity_same_dataset')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert abs(sum(SPLIT_RATIOS) - 1.0) < 1e-9
assert LABEL_GENUINE != LABEL_FORGED
print('Run:', RUN_VERSION)
print('Output:', OUT_DIR)

In [ ]:
# Cell 4 — read Kaggle Secrets without printing secret values
from kaggle_secrets import UserSecretsClient

_secret_client = UserSecretsClient()

def kaggle_secret(name):
    try:
        return _secret_client.get_secret(name)
    except Exception:
        return None

HF_TOKEN = kaggle_secret('HF_TOKEN')
COMET_API_KEY = kaggle_secret('COMET_API_KEY')
COMET_WORKSPACE = kaggle_secret('COMET_WORKSPACE')
AWS_ACCESS_KEY_ID = kaggle_secret('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = kaggle_secret('AWS_SECRET_ACCESS_KEY')
AWS_SESSION_TOKEN = kaggle_secret('AWS_SESSION_TOKEN')
AWS_REGION = kaggle_secret('AWS_REGION') or kaggle_secret('AWS_DEFAULT_REGION') or 'ap-southeast-2'
AWS_S3_BUCKET = kaggle_secret('AWS_S3_BUCKET')

COMET_ENABLED = bool(USE_COMET and COMET_API_KEY)
S3_ENABLED = bool(UPLOAD_TO_S3 and AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and AWS_S3_BUCKET)
print({
    'HF token configured': bool(HF_TOKEN),
    'Comet enabled': COMET_ENABLED,
    'S3 enabled': S3_ENABLED,
    'AWS region': AWS_REGION,
})
if USE_COMET and not COMET_ENABLED:
    print('Comet skipped: add COMET_API_KEY to Kaggle Secrets.')
if UPLOAD_TO_S3 and not S3_ENABLED:
    print('S3 skipped: add AWS credentials and AWS_S3_BUCKET to Kaggle Secrets.')

## Load the native verification pairs

The detection ZIP contains document images and boxes. Similarity needs signer evidence, so this notebook uses the native pair representation of the underlying corpus rather than inventing labels from boxes. The Hugging Face `main` revision is resolved to a commit SHA for reproducibility.

In [ ]:
# Cell 5 — resolve immutable revision, download all available splits, validate schema
from datasets import load_dataset
from huggingface_hub import HfApi

DATASET_COMMIT = HfApi(token=HF_TOKEN).dataset_info(HF_PAIR_DATASET, revision=HF_REVISION).sha
RAW_DATA = load_dataset(HF_PAIR_DATASET, revision=DATASET_COMMIT, token=HF_TOKEN)
print('Dataset commit:', DATASET_COMMIT)
print(RAW_DATA)

required_columns = {'sample_signature', 'to_verify_signature', 'label'}
identity_candidates = {'writer_id', 'writer', 'signer_id', 'signer', 'individual', 'person_id'}
for split_name, split_data in RAW_DATA.items():
    missing = required_columns - set(split_data.column_names)
    assert not missing, f'{split_name} is missing columns: {sorted(missing)}'

# Keep only pair-training fields so large document images and detection boxes are not decoded per batch.
PAIR_DATA = {
    split_name: split_data.select_columns(sorted(required_columns | (set(split_data.column_names) & identity_candidates)))
    for split_name, split_data in RAW_DATA.items()
}
all_labels = Counter()
for split_data in PAIR_DATA.values():
    all_labels.update(int(value) for value in split_data['label'])
print('Raw labels:', dict(sorted(all_labels.items())))
print('Training columns:', {name: data.column_names for name, data in PAIR_DATA.items()})
assert set(all_labels) == {LABEL_GENUINE, LABEL_FORGED}, 'Inspect label semantics before continuing.'

In [ ]:
# Cell 6 — visual label sanity check: 0 should be genuine, 1 should be forged
def first_row_with_label(wanted):
    for split_name, split_data in PAIR_DATA.items():
        for index, value in enumerate(split_data['label']):
            if int(value) == wanted:
                return split_name, index, split_data[index]
    raise LookupError(wanted)

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for row_number, label in enumerate((LABEL_GENUINE, LABEL_FORGED)):
    split_name, row_index, row = first_row_with_label(label)
    axes[row_number, 0].imshow(row['sample_signature'].convert('L'), cmap='gray')
    axes[row_number, 0].set_title(f'Reference | label={label}')
    axes[row_number, 1].imshow(row['to_verify_signature'].convert('L'), cmap='gray')
    axes[row_number, 1].set_title(f'Candidate | label={label}')
    for axis in axes[row_number]:
        axis.axis('off')
plt.suptitle('Manual label check: top genuine, bottom forged')
plt.tight_layout()
plt.savefig(OUT_DIR / 'label_sanity_check.png', dpi=160, bbox_inches='tight')
plt.show()
print('Before training, visually confirm that label 0 looks genuine and label 1 looks forged.')

## Leakage-aware split

If the dataset exposes a writer/signer identity column, the split is writer-disjoint. Otherwise, all rows containing the same normalized reference-signature fingerprint stay in one split. This fallback blocks exact-reference leakage but is not equivalent to a guaranteed writer-disjoint evaluation.

In [ ]:
# Cell 7 — build row index and deterministic group split
IDENTITY_CANDIDATES = ('writer_id', 'writer', 'signer_id', 'signer', 'individual', 'person_id')
common_columns = set.intersection(*(set(split.column_names) for split in PAIR_DATA.values()))
identity_field = next((name for name in IDENTITY_CANDIDATES if name in common_columns), None)

def normalized_reference_fingerprint(image):
    gray = ImageOps.autocontrast(image.convert('L')).resize((128, 64), Image.Resampling.BILINEAR)
    return hashlib.sha256(np.asarray(gray, dtype=np.uint8).tobytes()).hexdigest()

def assigned_split(group_key):
    digest = hashlib.sha256(f'{SEED}:{group_key}'.encode('utf-8')).hexdigest()
    position = int(digest[:12], 16) / float(16**12)
    train_edge = SPLIT_RATIOS[0]
    val_edge = SPLIT_RATIOS[0] + SPLIT_RATIOS[1]
    return 'train' if position < train_edge else 'val' if position < val_edge else 'test'

records = []
for source_split, split_data in PAIR_DATA.items():
    for row_index in tqdm(range(len(split_data)), desc=f'Indexing {source_split}'):
        row = split_data[row_index]
        if identity_field:
            group = f'writer:{row[identity_field]}'
        else:
            group = 'reference:' + normalized_reference_fingerprint(row['sample_signature'])
        records.append({
            'source_split': source_split,
            'row_index': row_index,
            'label': int(row['label']),
            'group': group,
            'split': assigned_split(group),
        })

SPLIT_RECORDS = {name: [record for record in records if record['split'] == name] for name in ('train', 'val', 'test')}
SPLIT_GROUPS = {name: {record['group'] for record in rows} for name, rows in SPLIT_RECORDS.items()}
assert SPLIT_GROUPS['train'].isdisjoint(SPLIT_GROUPS['val'])
assert SPLIT_GROUPS['train'].isdisjoint(SPLIT_GROUPS['test'])
assert SPLIT_GROUPS['val'].isdisjoint(SPLIT_GROUPS['test'])
for split_name, split_records in SPLIT_RECORDS.items():
    labels = Counter(record['label'] for record in split_records)
    assert set(labels) == {LABEL_GENUINE, LABEL_FORGED}, f'{split_name} does not contain both labels.'
    print(split_name, '| rows:', len(split_records), '| groups:', len(SPLIT_GROUPS[split_name]), '| labels:', dict(labels))

SPLIT_MODE = f'writer-disjoint:{identity_field}' if identity_field else 'exact-reference-fingerprint-disjoint'
print('Leakage gate: PASS | mode:', SPLIT_MODE)

## API-compatible preprocessing and crop-noise augmentation

Validation and test exactly match `SignatureEmbedder._preprocess`: grayscale, autocontrast, white square padding, 224×224 resize, RGB conversion, and ImageNet normalization. Training adds only mild geometric/photometric noise before that same preprocessing to simulate detector-crop variation.

In [ ]:
# Cell 8 — preprocessing, Dataset objects and balanced DataLoader
MEAN = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
STD = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)

train_augmentation = torchvision.transforms.Compose([
    torchvision.transforms.RandomAffine(degrees=4, translate=(0.04, 0.04), scale=(0.92, 1.08), fill=255),
    torchvision.transforms.RandomApply([torchvision.transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))], p=0.15),
])

def api_preprocess(image, augment=False):
    gray = ImageOps.autocontrast(image.convert('L'))
    if augment:
        gray = train_augmentation(gray)
    side = max(gray.size)
    canvas = Image.new('L', (side, side), 255)
    canvas.paste(gray, ((side - gray.width) // 2, (side - gray.height) // 2))
    canvas = canvas.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR).convert('RGB')
    array = np.asarray(canvas, dtype=np.uint8).copy()
    tensor = torch.from_numpy(array).permute(2, 0, 1).float().div(255.0)
    return (tensor - MEAN) / STD

class SignaturePairDataset(torch.utils.data.Dataset):
    def __init__(self, split_name, augment=False):
        self.records = SPLIT_RECORDS[split_name]
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        row = PAIR_DATA[record['source_split']][record['row_index']]
        left = api_preprocess(row['sample_signature'], self.augment)
        right = api_preprocess(row['to_verify_signature'], self.augment)
        is_match = 1 if record['label'] == LABEL_GENUINE else 0
        return left, right, torch.tensor(is_match, dtype=torch.long)

train_dataset = SignaturePairDataset('train', augment=True)
val_dataset = SignaturePairDataset('val')
test_dataset = SignaturePairDataset('test')
train_counts = Counter(record['label'] for record in SPLIT_RECORDS['train'])
sample_weights = [1.0 / train_counts[record['label']] for record in SPLIT_RECORDS['train']]
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': device.type == 'cuda',
    'persistent_workers': NUM_WORKERS > 0,
}
train_loader = torch.utils.data.DataLoader(train_dataset, sampler=sampler, **loader_kwargs)
val_loader = torch.utils.data.DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False, **loader_kwargs)
print('Batches:', {'train': len(train_loader), 'val': len(val_loader), 'test': len(test_loader)})

In [ ]:
# Cell 9 — model and verification metrics
def make_backbone(pretrained=True):
    weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = torch.nn.Identity()
    return model

def metrics_at_threshold(scores, labels, threshold):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    predictions = scores >= float(threshold)
    positives = labels == 1
    negatives = labels == 0
    far = float((predictions & negatives).sum() / max(1, negatives.sum()))
    frr = float(((~predictions) & positives).sum() / max(1, positives.sum()))
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions.astype(int), average='binary', zero_division=0)
    return {
        'threshold_used': float(threshold),
        'auc': float(roc_auc_score(labels, scores)),
        'far': far,
        'frr': frr,
        'accuracy': float((predictions == labels).mean()),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'pairs': int(len(labels)),
    }

def select_eer_threshold(scores, labels):
    fpr, tpr, thresholds = roc_curve(labels, scores)
    finite = np.isfinite(thresholds)
    fpr, tpr, thresholds = fpr[finite], tpr[finite], thresholds[finite]
    fnr = 1.0 - tpr
    index = int(np.argmin(np.abs(fpr - fnr)))
    return float(thresholds[index]), float((fpr[index] + fnr[index]) / 2.0)

@torch.inference_mode()
def collect_scores(model, loader):
    model.eval()
    scores, labels = [], []
    for left, right, batch_labels in tqdm(loader, desc='Evaluating', leave=False):
        left = left.to(device, non_blocking=True)
        right = right.to(device, non_blocking=True)
        left_embedding = torch.nn.functional.normalize(model(left), dim=1)
        right_embedding = torch.nn.functional.normalize(model(right), dim=1)
        scores.extend(torch.nn.functional.cosine_similarity(left_embedding, right_embedding).cpu().tolist())
        labels.extend(batch_labels.tolist())
    return np.asarray(scores), np.asarray(labels)

In [ ]:
# Cell 10 — ImageNet ResNet18 baseline on validation and test
baseline_model = make_backbone(PRETRAINED).to(device).eval()
baseline_val_scores, val_labels = collect_scores(baseline_model, val_loader)
BASELINE_THRESHOLD, baseline_val_eer = select_eer_threshold(baseline_val_scores, val_labels)
baseline_val = metrics_at_threshold(baseline_val_scores, val_labels, BASELINE_THRESHOLD)
baseline_test_scores, test_labels = collect_scores(baseline_model, test_loader)
baseline_test = metrics_at_threshold(baseline_test_scores, test_labels, BASELINE_THRESHOLD)
print('Baseline validation EER:', round(baseline_val_eer, 4))
print('Baseline test:', json.dumps(baseline_test, indent=2))
del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Cell 11 — optional Comet experiment
comet_experiment = None
if COMET_ENABLED:
    import comet_ml
    comet_experiment = comet_ml.start(
        api_key=COMET_API_KEY,
        workspace=COMET_WORKSPACE or None,
        project_name=COMET_PROJECT,
        mode='create',
        experiment_config=comet_ml.ExperimentConfig(
            name=RUN_VERSION,
            tags=['kaggle', 'similarity', 'siamese', 'same-dataset', 'resnet18'],
        ),
    )
    comet_experiment.log_parameters({
        'dataset': HF_PAIR_DATASET,
        'dataset_commit': DATASET_COMMIT,
        'split_mode': SPLIT_MODE,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'cosine_margin': COSINE_MARGIN,
        'image_size': IMG_SIZE,
        'seed': SEED,
    })
    comet_experiment.log_metrics({'baseline_val_eer': baseline_val_eer, 'baseline_test_auc': baseline_test['auc']})
    print('Comet experiment:', comet_experiment.url)
else:
    print('Comet disabled; local artifacts will still be generated.')

## Siamese fine-tuning

`CosineEmbeddingLoss` directly optimizes the same cosine similarity used by the API. Genuine pairs receive target `+1`; forged pairs receive target `-1`. Balanced sampling prevents the majority label from dominating. Early stopping sees validation data only.

In [ ]:
# Cell 12 — train with mixed precision and validation-only early stopping
model = make_backbone(PRETRAINED).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = torch.nn.CosineEmbeddingLoss(margin=COSINE_MARGIN)
amp_enabled = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

best_eer = float('inf')
best_epoch = 0
bad_epochs = 0
history = []
best_path = OUT_DIR / 'siamese_resnet18_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    started = time.time()
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', unit='batch')
    for left, right, is_match in progress:
        left = left.to(device, non_blocking=True)
        right = right.to(device, non_blocking=True)
        target = torch.where(is_match.to(device) == 1, 1.0, -1.0)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
            left_embedding = torch.nn.functional.normalize(model(left), dim=1)
            right_embedding = torch.nn.functional.normalize(model(right), dim=1)
            loss = criterion(left_embedding, right_embedding, target)
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss in epoch {epoch}.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))
        progress.set_postfix(loss=f'{losses[-1]:.4f}')

    scheduler.step()
    val_scores, val_labels = collect_scores(model, val_loader)
    val_threshold, val_eer = select_eer_threshold(val_scores, val_labels)
    val_metrics = metrics_at_threshold(val_scores, val_labels, val_threshold)
    row = {
        'epoch': epoch,
        'train_loss': float(np.mean(losses)),
        'val_eer': val_eer,
        'val_auc': val_metrics['auc'],
        'val_far': val_metrics['far'],
        'val_frr': val_metrics['frr'],
        'val_threshold': val_threshold,
        'learning_rate': float(scheduler.get_last_lr()[0]),
        'seconds': round(time.time() - started, 1),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUT_DIR / 'siamese_history.csv', index=False)
    print(row)
    if comet_experiment:
        comet_experiment.log_metrics({key: value for key, value in row.items() if key != 'epoch'}, step=epoch, epoch=epoch)

    if val_eer < best_eer - EARLY_STOPPING_MIN_DELTA:
        best_eer = val_eer
        best_epoch = epoch
        bad_epochs = 0
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, best_path)
        print(f'New best: epoch={epoch}, val EER={val_eer:.4f}')
    else:
        bad_epochs += 1
        print(f'No improvement: {bad_epochs}/{EARLY_STOPPING_PATIENCE}')
        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print('Early stopping. Best epoch:', best_epoch)
            break

assert best_path.exists(), 'Training did not produce a checkpoint.'

In [ ]:
# Cell 13 — choose threshold on validation, then evaluate test once
best_state = torch.load(best_path, map_location='cpu', weights_only=True)
model.load_state_dict(best_state)
model.to(device).eval()

siamese_val_scores, val_labels = collect_scores(model, val_loader)
SIMILARITY_THRESHOLD, siamese_val_eer = select_eer_threshold(siamese_val_scores, val_labels)
siamese_val = metrics_at_threshold(siamese_val_scores, val_labels, SIMILARITY_THRESHOLD)
siamese_test_scores, test_labels = collect_scores(model, test_loader)
siamese_test = metrics_at_threshold(siamese_test_scores, test_labels, SIMILARITY_THRESHOLD)
_, test_eer = select_eer_threshold(siamese_test_scores, test_labels)
siamese_test['eer_for_reporting_only'] = test_eer

comparison = pd.DataFrame([
    {'method': 'ImageNet ResNet18 baseline', 'val_eer': baseline_val_eer, **{f'test_{key}': value for key, value in baseline_test.items()}},
    {'method': 'Siamese ResNet18', 'val_eer': siamese_val_eer, **{f'test_{key}': value for key, value in siamese_test.items()}},
])
comparison.to_csv(OUT_DIR / 'similarity_comparison.csv', index=False)
display(comparison.round(4))
print('Recommended API threshold:', round(SIMILARITY_THRESHOLD, 6))

In [ ]:
# Cell 14 — ROC, score distribution and training curves
history_frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, scores in [('Baseline', baseline_test_scores), ('Siamese', siamese_test_scores)]:
    fpr, tpr, _ = roc_curve(test_labels, scores)
    axes[0].plot(fpr, tpr, label=f'{name} AUC={roc_auc_score(test_labels, scores):.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set(xlabel='False accept rate', ylabel='True accept rate', title='Held-out test ROC')
axes[0].legend(); axes[0].grid(alpha=0.25)

axes[1].hist(siamese_test_scores[test_labels == 1], bins=40, alpha=0.6, density=True, label='genuine')
axes[1].hist(siamese_test_scores[test_labels == 0], bins=40, alpha=0.6, density=True, label='forged')
axes[1].axvline(SIMILARITY_THRESHOLD, color='black', linestyle='--', label=f'threshold={SIMILARITY_THRESHOLD:.3f}')
axes[1].set(xlabel='Cosine similarity', title='Test score distribution')
axes[1].legend(); axes[1].grid(alpha=0.25)

axes[2].plot(history_frame.epoch, history_frame.train_loss, marker='o', label='train loss')
axes[2].plot(history_frame.epoch, history_frame.val_eer, marker='o', label='validation EER')
axes[2].set(xlabel='Epoch', title='Training history')
axes[2].legend(); axes[2].grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / 'similarity_results.png', dpi=180, bbox_inches='tight')
plt.show()

## Export for the existing API and S3 registry

The service expects a plain torchvision ResNet18 state dict whose `fc` is `Identity`. The alias must contain `version`, `weights_key`, `sha256`, and `recommended_threshold`. The immutable version is uploaded first; `latest` is changed last.

In [ ]:
# Cell 15 — API compatibility gate and local artifacts
weights_file = OUT_DIR / 'siamese_resnet18.pt'
torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, weights_file)

compatibility_model = make_backbone(pretrained=False)
compatibility_state = torch.load(weights_file, map_location='cpu', weights_only=True)
missing, unexpected = compatibility_model.load_state_dict(compatibility_state, strict=False)
assert not missing and not unexpected, f'API mismatch: missing={missing}, unexpected={unexpected}'
with torch.inference_mode():
    probe = torch.nn.functional.normalize(compatibility_model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)), dim=1)
assert probe.shape == (1, 512) and torch.isfinite(probe).all()
print('API compatibility gate: PASS')

def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()

weights_sha256 = sha256_file(weights_file)
created_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
split_report = {
    name: {
        'rows': len(rows),
        'groups': len(SPLIT_GROUPS[name]),
        'genuine': sum(record['label'] == LABEL_GENUINE for record in rows),
        'forged': sum(record['label'] == LABEL_FORGED for record in rows),
    }
    for name, rows in SPLIT_RECORDS.items()
}
report = {
    'schema_version': 1,
    'version': RUN_VERSION,
    'created_at_utc': created_at,
    'task': 'offline signature verification',
    'dataset': HF_PAIR_DATASET,
    'dataset_commit': DATASET_COMMIT,
    'split_mode': SPLIT_MODE,
    'split_report': split_report,
    'split_limitation': None if identity_field else 'Exact reference images are disjoint, but complete writer-disjointness cannot be proven without writer IDs.',
    'label_semantics': {'0': 'genuine', '1': 'forged'},
    'seed': SEED,
    'best_epoch': best_epoch,
    'best_validation_eer': best_eer,
    'recommended_threshold': SIMILARITY_THRESHOLD,
    'validation': siamese_val,
    'test': siamese_test,
    'baseline_test': baseline_test,
    'weights_file': weights_file.name,
    'weights_sha256': weights_sha256,
    'preprocessing': 'API exact: grayscale, autocontrast, white square pad, 224x224 bilinear, ImageNet normalization',
}
report_file = OUT_DIR / 'similarity_report.json'
report_file.write_text(json.dumps(report, indent=2), encoding='utf-8')
manifest = {
    'schema_version': 1,
    'version': RUN_VERSION,
    'task': 'signature_similarity',
    'weights_key': None,
    'weights_sha256': weights_sha256,
    'weights_size_bytes': weights_file.stat().st_size,
    'recommended_threshold': SIMILARITY_THRESHOLD,
    'best_epoch': best_epoch,
    'test_auc': siamese_test['auc'],
    'test_eer': siamese_test['eer_for_reporting_only'],
    'test_far': siamese_test['far'],
    'test_frr': siamese_test['frr'],
    'created_at_utc': created_at,
}
manifest_file = OUT_DIR / 'manifest.json'
manifest_file.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps({'version': RUN_VERSION, 'threshold': SIMILARITY_THRESHOLD, 'sha256': weights_sha256}, indent=2))

In [ ]:
# Cell 16 — upload immutable version, verify weights, update latest alias last
S3_WEIGHTS_URI = None
if S3_ENABLED:
    import boto3
    from botocore.config import Config
    from botocore.exceptions import ClientError

    client_kwargs = {
        'aws_access_key_id': AWS_ACCESS_KEY_ID,
        'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
        'region_name': AWS_REGION,
        'config': Config(retries={'mode': 'standard', 'max_attempts': 5}),
    }
    if AWS_SESSION_TOKEN:
        client_kwargs['aws_session_token'] = AWS_SESSION_TOKEN
    s3 = boto3.client('s3', **client_kwargs)
    version_prefix = f"{S3_PREFIX.strip('/')}/versions/{RUN_VERSION}"
    weights_key = f'{version_prefix}/siamese_resnet18.pt'

    try:
        s3.head_object(Bucket=AWS_S3_BUCKET, Key=weights_key)
    except ClientError as error:
        if error.response.get('Error', {}).get('Code') not in {'404', 'NoSuchKey', 'NotFound', '403', 'AccessDenied'}:
            raise
    else:
        raise FileExistsError(f'Immutable model version already exists: {RUN_VERSION}')

    s3.upload_file(str(weights_file), AWS_S3_BUCKET, weights_key, ExtraArgs={
        'ServerSideEncryption': 'AES256',
        'Metadata': {'sha256': weights_sha256, 'model-version': RUN_VERSION},
    })
    head = s3.head_object(Bucket=AWS_S3_BUCKET, Key=weights_key)
    assert head['ContentLength'] == weights_file.stat().st_size
    assert head.get('Metadata', {}).get('sha256') == weights_sha256

    artifacts = [report_file, manifest_file, OUT_DIR / 'siamese_history.csv', OUT_DIR / 'similarity_comparison.csv', OUT_DIR / 'similarity_results.png', OUT_DIR / 'label_sanity_check.png']
    for artifact in artifacts:
        content_type = 'application/json' if artifact.suffix == '.json' else 'text/csv' if artifact.suffix == '.csv' else 'image/png'
        s3.upload_file(str(artifact), AWS_S3_BUCKET, f'{version_prefix}/{artifact.name}', ExtraArgs={'ServerSideEncryption': 'AES256', 'ContentType': content_type})

    manifest['weights_key'] = weights_key
    manifest['s3_uri'] = f's3://{AWS_S3_BUCKET}/{weights_key}'
    manifest_file.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    s3.put_object(Bucket=AWS_S3_BUCKET, Key=f'{version_prefix}/manifest.json', Body=manifest_file.read_bytes(), ContentType='application/json', ServerSideEncryption='AES256')

    alias = {
        'version': RUN_VERSION,
        'weights_key': weights_key,
        'manifest_key': f'{version_prefix}/manifest.json',
        'sha256': weights_sha256,
        'recommended_threshold': SIMILARITY_THRESHOLD,
        'test_auc': siamese_test['auc'],
        'test_eer': siamese_test['eer_for_reporting_only'],
        'updated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }
    s3.put_object(Bucket=AWS_S3_BUCKET, Key=f"{S3_PREFIX.strip('/')}/aliases/{S3_ALIAS}.json", Body=json.dumps(alias, indent=2).encode('utf-8'), ContentType='application/json', ServerSideEncryption='AES256')
    S3_WEIGHTS_URI = f's3://{AWS_S3_BUCKET}/{weights_key}'
    print('S3 verified:', S3_WEIGHTS_URI)
    print('Alias updated:', f's3://{AWS_S3_BUCKET}/{S3_PREFIX}/aliases/{S3_ALIAS}.json')
else:
    print('S3 upload skipped. Artifacts remain in:', OUT_DIR)

In [ ]:
# Cell 17 — finish Comet and create a downloadable Kaggle bundle
if comet_experiment:
    comet_experiment.log_metrics({
        'best_epoch': best_epoch,
        'recommended_threshold': SIMILARITY_THRESHOLD,
        'test_auc': siamese_test['auc'],
        'test_eer': siamese_test['eer_for_reporting_only'],
        'test_far': siamese_test['far'],
        'test_frr': siamese_test['frr'],
        'test_accuracy': siamese_test['accuracy'],
        'baseline_test_auc': baseline_test['auc'],
    })
    comet_experiment.log_table('similarity_comparison.csv', tabular_data=comparison)
    comet_experiment.log_image(str(OUT_DIR / 'similarity_results.png'), name='similarity_results')
    comet_experiment.log_image(str(OUT_DIR / 'label_sanity_check.png'), name='label_sanity_check')
    comet_experiment.log_asset(str(report_file))
    comet_experiment.log_asset(str(manifest_file))
    if S3_WEIGHTS_URI:
        comet_experiment.log_other('s3_weights_uri', S3_WEIGHTS_URI)
    comet_experiment.end()

bundle_base = Path('/kaggle/working') / f'signature_similarity_same_dataset_{RUN_VERSION}'
bundle_path = shutil.make_archive(str(bundle_base), 'zip', root_dir=OUT_DIR)
print('\nCOMPLETE')
print('Version:', RUN_VERSION)
print('Split mode:', SPLIT_MODE)
print('Best epoch:', best_epoch)
print('Test AUC:', round(siamese_test['auc'], 4))
print('Test EER:', round(siamese_test['eer_for_reporting_only'], 4))
print('Test FAR / FRR:', round(siamese_test['far'], 4), '/', round(siamese_test['frr'], 4))
print('Recommended API threshold:', round(SIMILARITY_THRESHOLD, 6))
print('Weights:', weights_file)
print('Bundle:', bundle_path)
print('S3:', S3_WEIGHTS_URI or 'not uploaded')
print('\nEC2 remains configured with:')
print('SIMILARITY_S3_PREFIX=signatureai/similarity')
print('SIMILARITY_S3_ALIAS=latest')